In [37]:
import pandas as pd 

df = pd.read_csv('Zomato.csv' , encoding = 'latin1')

df.head(2)
df.info()
df.isnull().sum()
df.head(2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Restaurant ID         9551 non-null   int64  
 1   Restaurant Name       9551 non-null   object 
 2   Country Code          9551 non-null   int64  
 3   City                  9551 non-null   object 
 4   Address               9551 non-null   object 
 5   Locality              9551 non-null   object 
 6   Locality Verbose      9551 non-null   object 
 7   Longitude             9551 non-null   float64
 8   Latitude              9551 non-null   float64
 9   Cuisines              9542 non-null   object 
 10  Average Cost for two  9551 non-null   int64  
 11  Currency              9551 non-null   object 
 12  Has Table booking     9551 non-null   object 
 13  Has Online delivery   9551 non-null   object 
 14  Is delivering now     9551 non-null   object 
 15  Switch to order menu 

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591


In [38]:
import pandas as pd 
import numpy as np

df = pd.read_csv('Zomato.csv' , encoding = 'latin1')

print("Missing values:")
print(df[['Average Cost for two', 'Aggregate rating']].isnull().sum())

df['Cuisines'] = df['Cuisines'].fillna(df['Cuisines'].mode()[0])

Q1 = df['Average Cost for two'].quantile(0.25)
Q3 = df['Average Cost for two'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print("Outliers found:", ((df['Average Cost for two'] < lower) | (df['Average Cost for two'] > upper)).sum())

# Cap outliers
df['Average Cost for two'] = df['Average Cost for two'].clip(lower, upper)




Missing values:
Average Cost for two    0
Aggregate rating        0
dtype: int64
Outliers found: 853


In [39]:
from sklearn.preprocessing import StandardScaler 
from sklearn.preprocessing import LabelEncoder

le_cuisine = LabelEncoder()
df['Cuisines_encoded'] = le_cuisine.fit_transform(df['Cuisines'].astype(str))

le_city = LabelEncoder()
df['City_encoded'] = le_city.fit_transform(df['City'])

scaler = StandardScaler()
df[['Average Cost for two', 'Aggregate rating']] = scaler.fit_transform(
    df[['Average Cost for two', 'Aggregate rating']])


print(df[['Cuisines_encoded', 'City_encoded', 'Average Cost for two', 'Aggregate rating']].head())
df.head(5)

   Cuisines_encoded  City_encoded  Average Cost for two  Aggregate rating
0               920            73              1.525039          1.407131
1              1111            73              1.788682          1.209281
2              1671            75              2.250058          1.143331
3              1126            75              2.250058          1.473081
4              1122            75              2.250058          1.407131


,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes,Cuisines_encoded,City_encoded
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,No,No,No,3,1.407131,Dark Green,Excellent,314,920,73
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,No,No,No,3,1.209281,Dark Green,Excellent,591,1111,73
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,No,No,No,4,1.143331,Green,Very Good,270,1671,75
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,No,No,No,4,1.473081,Dark Green,Excellent,365,1126,75
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,No,No,No,4,1.407131,Dark Green,Excellent,229,1122,75


In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score


df = pd.read_csv('Zomato.csv' , encoding = 'latin1')

df['Cuisines'] = df['Cuisines'].fillna(df['Cuisines'].mode()[0])


le_cuisine = LabelEncoder()
df['Cuisines_encoded'] = le_cuisine.fit_transform(df['Cuisines'].astype(str))

le_city = LabelEncoder()
df['City_encoded'] = le_city.fit_transform(df['City'])


df['high_rating'] = (df['Aggregate rating'] > 4.0).astype(int)

print("Aggregate rating range:", df['Aggregate rating'].min(), "to", df['Aggregate rating'].max())
print("Class distribution:\n", df['high_rating'].value_counts())

X = df[['Average Cost for two', 'Votes', 'Price range', 'Cuisines_encoded', 'City_encoded']]
y = df['high_rating']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)
auc_lr = roc_auc_score(y_test, lr.predict_proba(X_test_s)[:, 1])

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train_s, y_train)
auc_rf = roc_auc_score(y_test, rf.predict_proba(X_test_s)[:, 1])

print(f"\nLogistic Regression ROC-AUC: {auc_lr:.4f}")
print(f"Random Forest ROC-AUC      : {auc_rf:.4f}")
print(f"\nBetter model: {'Random Forest' if auc_rf > auc_lr else 'Logistic Regression'}")

Aggregate rating range: 0.0 to 4.9
Class distribution:
 high_rating
0    8437
1    1114
Name: count, dtype: int64

Logistic Regression ROC-AUC: 0.8782
Random Forest ROC-AUC      : 0.9322

Better model: Random Forest


In [47]:
df.columns

Index(['Restaurant ID', 'Restaurant Name', 'Country Code', 'City', 'Address',
       'Locality', 'Locality Verbose', 'Longitude', 'Latitude', 'Cuisines',
       'Average Cost for two', 'Currency', 'Has Table booking',
       'Has Online delivery', 'Is delivering now', 'Switch to order menu',
       'Price range', 'Aggregate rating', 'Rating color', 'Rating text',
       'Votes', 'Cuisines_encoded', 'City_encoded', 'high_rating'],
      dtype='object')